In [1]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

import datetime
import sys
import os
import warnings
from pathlib import Path
from typing import Any, Tuple, List, Dict
from dotmap import DotMap

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api
from aeon.schema.schemas import social02

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils import save_all_experiment_data, load_data_from_parquet

[2025-05-04 09:05:05,274][INFO]: Connecting apouget@aeon-db2:3306
[2025-05-04 09:05:05,296][INFO]: Connected apouget@aeon-db2:3306


# Definitions

In [2]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
os.makedirs(data_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas

In [3]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[0]

# Load data

In [4]:
def load_experiment_data(experiment, data_dir, periods=['presocial', 'social', 'postsocial'], 
                        data_types=['rfid', 'position'], trim_days=None):
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            df = load_data_from_parquet(
                experiment_name=experiment["name"],
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                else:  # position data
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'], keep='first')
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

# Load all periods for experiment
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['presocial', 'social', 'postsocial'],
    data_types=['rfid', 'position'],
    trim_days=4  # Optional: trim to 4 days
)

# Access data
presocial_rfid_df = data['presocial_rfid']
presocial_position_df = data['presocial_position']
social_rfid_df = data['social_rfid']
social_position_df = data['social_position']
postsocial_rfid_df = data['postsocial_rfid']
postsocial_position_df = data['postsocial_position']

Loading presocial rfid data...
Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_presocial_rfid.parquet...
  Loaded 912 rows
Combined data: 912 rows
  Trimmed to 4 days: 474 records
Loading presocial position data...
Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_presocial_position.parquet...
  Loaded 26653907 rows
Combined data: 26653907 rows
  Trimmed to 4 days: 13685319 records
  Removed duplicates: 13685319 -> 13241564
Loading social rfid data...
Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_social_rfid.parquet...
  Loaded 2010 rows
Combined data: 2010 rows
  Trimmed to 4 days: 582 records
Loading social position data...
Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_social_position.parquet...
  Loaded 106317415 rows
Combined data: 106317415 rows
  Trimmed to 4 days: 31266040 records

In [5]:
acquisition_computer = experiment["name"].split("-")[1].upper()
social_name = experiment["name"].split("-")[0]
metadata_root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
metadata_reader = social02.Metadata
metadata = aeon_api.load(metadata_root, metadata_reader)['metadata'].iloc[0]
rfid_devices_loc = DotMap({key: metadata.Devices[key].Location for key in metadata.Devices.keys() if 'rfid' in key.lower()})
rfid_devices_loc

DotMap(NestRfid1=DotMap(X='1218', Y='642'), NestRfid2=DotMap(X='1214', Y='426'), GateRfid=DotMap(X='195', Y='563'), Patch1Rfid=DotMap(X='940', Y='542'), Patch2Rfid=DotMap(X='600', Y='753'), Patch3Rfid=DotMap(X='589', Y='348'), _ipython_display_=DotMap(), _repr_mimebundle_=DotMap())

# Match RFID and Pose rows based on time and position

In [6]:
def match_rfid_to_pose(rfid_df, pose_df, rfid_devices_loc, cm2px, tolerance_ms=10):
    """
    Match RFID timestamps to nearest pose timestamps within tolerance.
    Returns dataframe with matched data, including unmatched RFID rows.
    """
    # Explode RFID data to have one row per timestamp/identity
    rfid_exploded = (
        rfid_df[['rfid_reader_name', 'timestamps', 'rfid']]
        .explode(['timestamps', 'rfid'])
        .rename(columns={'timestamps': 'rfid_time', 'rfid': 'rfid_identity'})
    )
    rfid_exploded['rfid_time'] = pd.to_datetime(rfid_exploded['rfid_time'])
    rfid_exploded['row_id'] = range(len(rfid_exploded))
    
    # Drop rows with null time values before merging
    rfid_valid = rfid_exploded.dropna(subset=['rfid_time']).copy()
    
    # Get RFID reader coordinates
    reader_coords = {}
    for name, loc in rfid_devices_loc.items():
        if (name.startswith('_') or 
            name.startswith('*') or 
            callable(getattr(rfid_devices_loc, name, None)) or
            name in ['to_pandas', 'to_dict', 'toDict', 'empty', 'copy']):
            continue
        if hasattr(loc, 'X') and hasattr(loc, 'Y'):
            reader_coords[name] = (float(loc.X), float(loc.Y))
    
    # Add reader coordinates to rfid_valid
    rfid_valid['rfid_x'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[0])
    rfid_valid['rfid_y'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[1])
    
    # Prepare pose data for efficient merging
    pose_subset = (
        pose_df[['identity_name', 'identity_likelihood', 'x', 'y', 'likelihood']]
        .reset_index()
        .rename(columns={
            'identity_name': 'pose_identity', 
            'time': 'pose_time', 
            'x': 'pose_x', 
            'y': 'pose_y',
            'identity_likelihood': 'pose_identity_likelihood',
            'likelihood': 'pose_likelihood'
        })
    )
    
    # Merge RFID with all possible pose identities to find all candidates
    unique_identities = pose_subset['pose_identity'].unique()
    rfid_expanded = pd.concat([
        rfid_valid.assign(pose_identity=identity) 
        for identity in unique_identities
    ])
    
    # Use merge_asof to find nearest timestamps within tolerance
    merged = pd.merge_asof(
        rfid_expanded.sort_values('rfid_time'),
        pose_subset.sort_values('pose_time'),
        left_on='rfid_time',
        right_on='pose_time',
        by='pose_identity',
        direction='nearest',
        tolerance=pd.Timedelta(f'{tolerance_ms}ms')
    )
    
    # Calculate spatial distances between reader and animal positions in cm
    merged['rfid_pose_distance'] = np.sqrt(
        (merged['pose_x'] - merged['rfid_x'])**2 + 
        (merged['pose_y'] - merged['rfid_y'])**2
    ) / cm2px
    
    # Keep only the closest match for each RFID detection (by distance)
    idx_closest = merged.groupby('row_id')['rfid_pose_distance'].idxmin()
    valid_idx = idx_closest.dropna()
    
    # Start with all rfid_valid rows
    result = rfid_valid.set_index('row_id')

    # Create empty dataframe with correct structure from merged
    empty_template = merged[['pose_time', 'pose_identity', 'pose_identity_likelihood', 
                            'pose_x', 'pose_y', 'pose_likelihood', 'rfid_pose_distance']].iloc[:0]
    result = result.join(empty_template)
    
    # Update with matched data where available
    if len(valid_idx) > 0:
        matched_data = merged.loc[valid_idx].set_index('row_id')
        result.update(matched_data)
    
    result = result.reset_index()
    
    # Reorder columns
    column_order = [
        'rfid_time', 'rfid_reader_name', 'rfid_identity', 'rfid_x', 'rfid_y',
        'pose_time', 'pose_identity', 'pose_identity_likelihood', 'pose_x', 'pose_y', 
        'pose_likelihood', 'rfid_pose_distance'
    ]
    
    # Select columns that exist in the result
    existing_columns = [col for col in column_order if col in result.columns]
    final_result = result[existing_columns].reset_index(drop=True)
    
    return final_result

In [7]:
pre_post_social_rfid_df = pd.concat([presocial_rfid_df, postsocial_rfid_df], ignore_index=True)
pre_post_social_position_df = pd.concat([presocial_position_df, postsocial_position_df], ignore_index=False)
pre_post_social_matched_df = match_rfid_to_pose(pre_post_social_rfid_df, pre_post_social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
print(pre_post_social_matched_df["rfid_time"].isna().sum())
display(pre_post_social_matched_df)
social_matched_df = match_rfid_to_pose(social_rfid_df, social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
display(social_matched_df)

0


,rfid_time,rfid_reader_name,rfid_identity,rfid_x,rfid_y,pose_time,pose_identity,pose_identity_likelihood,pose_x,pose_y,pose_likelihood,rfid_pose_distance
0,2024-01-31 11:29:46.238368034,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:46.240,BAA-1104045,NaN,606.112793,752.671265,0.872971,75.922105
1,2024-01-31 11:29:47.005440235,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:47.000,BAA-1104045,NaN,605.470093,752.491211,0.921314,76.008216
2,2024-01-31 11:29:47.404255867,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:47.400,BAA-1104045,NaN,603.172729,752.482666,0.932503,76.381643
3,2024-01-31 11:29:47.833759785,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:47.840,BAA-1104045,NaN,603.043335,752.548523,0.939723,76.409457
4,2024-01-31 11:29:48.232607841,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:48.240,BAA-1104045,NaN,592.466797,760.374023,0.993504,78.932069
...,...,...,...,...,...,...,...,...,...,...,...,...
206414,2024-02-29 16:55:01.558495998,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-29 16:55:01.560,BAA-1104047,NaN,589.371887,368.192566,0.906623,3.883844
206415,2024-02-29 16:55:02.018591881,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-29 16:55:02.020,BAA-1104047,NaN,592.562012,368.317993,0.896735,3.966897
206416,2024-02-29 16:55:02.386720181,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-29 16:55:02.380,BAA-1104047,NaN,592.674316,368.700623,0.885106,4.043113
206417,2024-02-29 16:55:02.693568230,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-29 16:55:02.700,BAA-1104047,NaN,592.517090,370.865692,0.891463,4.448962


,rfid_time,rfid_reader_name,rfid_identity,rfid_x,rfid_y,pose_time,pose_identity,pose_identity_likelihood,pose_x,pose_y,pose_likelihood,rfid_pose_distance
0,2024-02-09 16:55:26.659776211,Patch1Rfid,BAA-1104045,940.0,542.0,2024-02-09 16:55:26.660,BAA-1104045,0.999980,987.512695,581.629272,0.917083,11.898137
1,2024-02-09 16:55:30.408192158,Patch1Rfid,BAA-1104045,940.0,542.0,2024-02-09 16:55:30.400,BAA-1104045,0.999953,936.765991,531.381653,0.965036,2.134599
2,2024-02-09 16:55:31.054719925,Patch1Rfid,BAA-1104045,940.0,542.0,2024-02-09 16:55:31.060,BAA-1104045,0.999928,925.598633,536.251465,0.963352,2.981979
3,2024-02-09 16:55:31.453567982,Patch1Rfid,BAA-1104045,940.0,542.0,2024-02-09 16:55:31.460,BAA-1104045,0.999916,923.269470,534.046448,0.973937,3.562469
4,2024-02-09 16:55:32.004896164,Patch1Rfid,BAA-1104045,940.0,542.0,2024-02-09 16:55:32.000,BAA-1104045,0.999970,923.065308,533.997498,0.917538,3.601979
...,...,...,...,...,...,...,...,...,...,...,...,...
187606,2024-02-13 15:21:33.675648212,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-13 15:21:33.680,BAA-1104047,0.999787,592.449036,373.420959,1.006673,4.933436
187607,2024-02-13 15:21:37.247744083,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-13 15:21:37.240,BAA-1104047,0.999767,616.220276,344.682251,0.962933,5.273408
187608,2024-02-13 15:21:47.704063892,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-13 15:21:47.700,BAA-1104047,0.998592,600.483643,347.057587,0.968547,2.215817
187609,2024-02-13 15:21:48.228896141,Patch3Rfid,BAA-1104047,589.0,348.0,2024-02-13 15:21:48.220,BAA-1104047,0.999679,594.935608,352.433563,0.959855,1.424738


# Calculate RFID readers' range

In [8]:
# Calculate RFID reader ranges
quantiles = [0.95, 0.99, 1.0]
reader_ranges = (
    pre_post_social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(reader_ranges)

RFID Reader Ranges (cm):
                  median  mean   p95   p99   max
rfid_reader_name                                
GateRfid             1.6   1.8   2.9   4.4  13.4
NestRfid1            2.9   3.6   7.9  10.0  12.7
NestRfid2            3.0   3.7   8.0   9.2  11.9
Patch1Rfid           4.1  14.4  75.3  79.0  84.0
Patch2Rfid          74.7  74.1  76.3  77.1  88.1
Patch3Rfid           3.6   3.5   4.5   4.6  12.9


In [9]:
# Optional debugging
import swc.aeon.io.reader
import swc.aeon.io.api
from aeon.schema.schemas import exp02
from aeon.analysis.movies import gridframes
from aeon.io.video import frames
from aeon.dj_pipeline.analysis.block_analysis import *
import plotly.express as px
# Plot the data on the corresponding video frame
idx = 1
fps = 50
time = pre_post_social_matched_df.iloc[idx]['pose_time']
df_to_plot = pre_post_social_matched_df.iloc[idx:idx+1]
display(df_to_plot)
root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
vid_data = swc.aeon.io.api.load(root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=1/fps/2))
vid_data = vid_data[time:time] # idk why this is necessary by for some reason the video data is loading more than just the ts between time and time+1/fps/2
fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))
# Add scatter plot with larger marker size
scatter_trace = px.scatter(df_to_plot, x='pose_x', y='pose_y', size_max=15).data[0]
scatter_trace2 = px.scatter(df_to_plot, x='rfid_x', y='rfid_y', size_max=15).data[0]
scatter_trace.marker.size = 6  # Increase marker size
scatter_trace.marker.color = 'red'  # Make markers more visible
fig.add_trace(scatter_trace)
fig.add_trace(scatter_trace2)
# Update layout to make plot bigger and adjust margins
fig.update_layout(
    width=1200,
    height=1000,
    margin=dict(l=20, r=20, t=20, b=20),  # Reduce margins to use more space
    showlegend=False
)
# Show the figure
fig.show()

,rfid_time,rfid_reader_name,rfid_identity,rfid_x,rfid_y,pose_time,pose_identity,pose_identity_likelihood,pose_x,pose_y,pose_likelihood,rfid_pose_distance
1,2024-01-31 11:29:47.005440235,Patch1Rfid,BAA-1104045,940.0,542.0,2024-01-31 11:29:47,BAA-1104045,NaN,605.470093,752.491211,0.921314,76.008216


In [10]:
# Calculate RFID reader ranges with social data 
# (this should be less accurate because a RFID reading could potentially be matched to the wrong subject if only 1 subject is detected by SLEAP)
quantiles = [0.95, 0.99, 1.0]
reader_ranges = (
    social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(reader_ranges)

RFID Reader Ranges (cm):
                  median  mean  p95  p99   max
rfid_reader_name                              
GateRfid             1.6   1.7  3.0  5.3  12.9
NestRfid1            2.8   3.4  7.1  9.1  12.7
NestRfid2            3.0   3.6  7.8  9.1  12.5
Patch1Rfid           3.4   3.2  4.0  5.3  12.3
Patch2Rfid           3.8   3.6  4.5  4.8  12.3
Patch3Rfid           3.9   3.6  4.5  4.8  13.6


# Assess SLEAP accuracy